# Diffusion 기반 데이터 증강 — baseline / conditional / joint / decision boundary

## 1. 모델 정의

In [1]:
import math
import os
from pathlib import Path # 파일, 폴더 관리
from functools import partial # 파라미터 고정
from tqdm.auto import tqdm

import torch
from torch import nn, einsum # 신경망 레이어, 행렬 곱
import torch.nn.functional as F # loss 계산, conv 연산
from torch.utils.data import DataLoader, Subset
from torch.optim import Adam

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from PIL import Image

from torchvision import transforms as T, utils
import torchvision.datasets as datasets
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix

from collections import defaultdict
import random

from torchvision.utils import save_image, make_grid
from pathlib import Path

from sklearn.manifold import TSNE
import seaborn as sns

from collections import Counter

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed) # cpu
    torch.cuda.manual_seed_all(seed) # gpu
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def exists(x):
    return x is not None # x가 None이냐 아니냐만 판단 T, F로 나옴

def default(val, d):
    if exists(val):
        return val
    return d() if callable(d) else d # callable 함수처럼 호출 가능하면 함수 실행

class Upsample(nn.Module):
    def __init__(self, dim, dim_out=None, scale_factor=2, mode="nearest"):
        super().__init__()
        self.scale_factor = scale_factor
        self.mode = mode
        self.conv = nn.Conv2d(dim, dim_out if dim_out is not None else dim, kernel_size=3, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=self.scale_factor, mode=self.mode) # (1, 64, 32, 32) → (1, 64, 64, 64)
        return self.conv(x)

class Downsample(nn.Module):
    def __init__(self, dim, dim_out=None):
        super().__init__()
        self.unshuffle = nn.PixelUnshuffle(2) # 공간 정보를 채널로 재배치, (B, C, H, W) → (B, C×4, H/2, W/2)
        self.conv = nn.Conv2d(dim * 4, dim_out or dim, kernel_size = 1) # 채널이 4배가 되기에

    def forward(self, x):
        return self.conv(self.unshuffle(x))

class Residual(nn.Module):  # 함수/레이어(fn) 를 감싸는(wrapper) 역할
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, *args, **kwargs): # fn이 x 말고 다른 입력도 받을 수 있게 하기 위함
        return self.fn(x, *args, **kwargs) + x # gradient가 절대 0이 안 됨

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :] # (B, 1) * (1, half_dim) = (B, half_dim)
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class PreNorm(nn.Module): # 어텐션 나온 피쳐맵, 활성값(activation) 정규화
    def __init__(self, dim, fn):
        super().__init__()
        self.fn = fn
        self.norm = nn.GroupNorm(1, dim) # nn.GroupNorm(num_groups, num_channels) # 그룹 하나로 모든 채널 평균,분산 계산

    def forward(self, x):
        return self.fn(self.norm(x))

class WeightStandardizedConv2d(nn.Conv2d): # 가중치 정규화
    def forward(self, x):
        eps = 1e-5 if x.dtype == torch.float32 else 1e-3
        w = self.weight # Conv2d의 상속 된 weight shape 가져오기, (out_channels, in_channels, kernel_h, kernel_w)
        mean = w.mean(dim=(1,2,3), keepdim=True) # (in_channels, kernel_h, kernel_w) 방향으로 평균, 차원 유지
        var = w.var(dim=(1,2,3), unbiased=False, keepdim=True)
        w = (w - mean) / torch.sqrt(var + eps) # 평균 ≈ 0, 분산 ≈ 1
        return F.conv2d(x, w, self.bias, self.stride, self.padding, self.dilation, self.groups)

class Block(nn.Module):
    def __init__(self, dim, dim_out, groups=8):
        super().__init__()
        self.proj = WeightStandardizedConv2d(dim, dim_out, kernel_size=3, padding=1) # 출력 채널 하나(필터 하나)를 구성하는 모든 가중치 (in_channels × kernel_h × kernel_w)
        self.norm = nn.GroupNorm(groups, dim_out) # Conv 결과로 나온 feature map(채널들)을 그룹 단위로 정규화하는 레이어
        self.act = nn.SiLU() # 부드러운 활성화 함수

    def forward(self, x, scale_shift=None):
        x = self.proj(x)
        x = self.norm(x)
        if exists(scale_shift):
            scale, shift = scale_shift
            x = x * (scale + 1) + shift # 학습 초기에 feature 붕괴 위험 -> scale + 1
        return self.act(x)

class ResnetBlock(nn.Module):
    def __init__(self, dim, dim_out, *, time_emb_dim=None, groups=8): # * 뒤에 오는 인자들은 키워드 인자로만 전달하라
        super().__init__()
        self.mlp = (
            nn.Sequential(
                nn.SiLU(),
                nn.Linear(time_emb_dim, dim_out * 2)
                )
            if exists(time_emb_dim) else None
        )

        self.block1 = Block(dim, dim_out, groups)
        self.block2 = Block(dim_out, dim_out, groups)
        self.res_conv = nn.Conv2d(dim, dim_out, 1) if dim != dim_out else nn.Identity()

    def forward(self, x, time_emb=None):
        scale_shift = None
        if exists(self.mlp) and exists(time_emb):
            h = self.mlp(time_emb)[:, :, None, None]
            scale_shift = h.chunk(2, dim=1)

        h = self.block1(x, scale_shift)
        h = self.block2(h)
        return h + self.res_conv(x)

class Attention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=32):
        super().__init__()
        self.heads = heads
        self.scale = dim_head ** -0.5 # 1 / sqrt(dim_head)
        hidden = heads * dim_head

        self.to_qkv = nn.Conv2d(dim, hidden * 3, 1, bias=False)
        self.to_out = nn.Conv2d(hidden, dim, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        q, k, v = self.to_qkv(x).chunk(3, dim=1)

        q = q.view(b, self.heads, -1, h * w) # -1 : 차원 자동 계산
        k = k.view(b, self.heads, -1, h * w)
        v = v.view(b, self.heads, -1, h * w)

        q = q * self.scale

        q_t = q.permute(0, 1, 3, 2)   # (B, H, N, D), 차원 재배치
        k_t = k                      # (B, H, D, N)

        attn = torch.softmax( # softmax는 attention score를 확률처럼 해석 가능한 가중치로 바꾸기 위해 필요
            torch.matmul(q_t, k_t),  # (B, H, N, N) 행렬 곱
            dim=-1
        )

        v_t = v.permute(0, 1, 3, 2)  # (B, H, N, D)

        out = torch.matmul(attn, v_t)   # (B, H, N, D)

        out = out.permute(0, 1, 3, 2)   # (B, H, D, N)
        out = out.contiguous().view(b, -1, h, w)

        return self.to_out(out)

class LinearAttention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=32):
        super().__init__()
        self.heads = heads
        self.scale = dim_head ** -0.5
        hidden = heads * dim_head

        self.to_qkv = nn.Conv2d(dim, hidden * 3, 1, bias=False)
        self.to_out = nn.Sequential(
            nn.Conv2d(hidden, dim, 1),
            nn.GroupNorm(1, dim)
        )

    def forward(self, x):
        b, c, h, w = x.shape
        q, k, v = self.to_qkv(x).chunk(3, dim=1)

        q = q.view(b, self.heads, -1, h * w).softmax(dim=-2) # (B, H, D, N) D 정규화
        k = k.view(b, self.heads, -1, h * w).softmax(dim=-1) # (B, H, D, N) N 정규화
        v = v.view(b, self.heads, -1, h * w)

        q = q * self.scale

        v_t = v.permute(0, 1, 3, 2)        # (B, H, N, D)
        context = torch.matmul(k, v_t)     # (B, H, D, D)

        context_t = context.permute(0, 1, 3, 2)  # (B, H, D, D)
        out = torch.matmul(context_t, q)         # (B, H, D, N)

        out = out.contiguous().view(b, -1, h, w) # contiguous = 메모리 정렬
        return self.to_out(out)

class Unet(nn.Module):
    def __init__(self, dim, init_dim=None, out_dim=None, dim_mults=(1, 2, 4), channels=3, resnet_block_groups=4, num_classes=3):
        super().__init__()
        self.channels = channels
        init_dim = default(init_dim, dim) # 28(이미지 크기)
        self.init_conv = nn.Conv2d(channels, init_dim, 7, padding=3)

        dims = [init_dim, *map(lambda m: dim * m, dim_mults)] # 예시 [64, 64, 128, 256]
        in_out = list(zip(dims[:-1], dims[1:])) # 예시 [(64,64), (64,128), (128,256)]
        block_klass = partial(ResnetBlock, groups=resnet_block_groups) # 함수(또는 클래스)에 일부 인자를 미리 고정

        time_dim = dim * 4
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(dim),
            nn.Linear(dim, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, time_dim),
        )
        self.label_emb = nn.Embedding(num_classes, time_dim)

        self.downs = nn.ModuleList([])
        self.ups = nn.ModuleList([])
        for ind, (dim_in, dim_out) in enumerate(in_out): # 인덱스, (64, 64)
            is_last = ind >= (len(in_out) - 1)
            self.downs.append(nn.ModuleList([
                block_klass(dim_in, dim_in, time_emb_dim=time_dim),
                block_klass(dim_in, dim_in, time_emb_dim=time_dim),
                Residual(PreNorm(dim_in, LinearAttention(dim_in))),
                Downsample(dim_in, dim_out) if not is_last else nn.Conv2d(dim_in, dim_out, 3, padding=1)
            ]))

        mid_dim = dims[-1]
        self.mid_block1 = block_klass(mid_dim, mid_dim, time_emb_dim=time_dim)
        self.mid_attn = Residual(PreNorm(mid_dim, Attention(mid_dim)))
        self.mid_block2 = block_klass(mid_dim, mid_dim, time_emb_dim=time_dim)

        for ind, (dim_in, dim_out) in enumerate(reversed(in_out)):
            is_last = ind == (len(in_out) - 1)
            self.ups.append(nn.ModuleList([
                block_klass(dim_out + dim_in, dim_out, time_emb_dim=time_dim),
                block_klass(dim_out + dim_in, dim_out, time_emb_dim=time_dim),
                Residual(PreNorm(dim_out, LinearAttention(dim_out))),
                Upsample(dim_out, dim_in) if not is_last else nn.Conv2d(dim_out, dim_in, 3, padding=1)
            ]))

        self.final_res_block = block_klass(dim * 2, dim, time_emb_dim=time_dim)
        self.final_conv = nn.Conv2d(dim, channels, 1)

    def forward(self, x, time, y):
        x = self.init_conv(x)
        r = x.clone()
        t = self.time_mlp(time)
        y_emb = self.label_emb(y)

        t = t + y_emb

        h = []
        for block1, block2, attn, downsample in self.downs:
            x = block1(x, t)
            h.append(x)
            x = block2(x, t)
            x = attn(x)
            h.append(x)
            x = downsample(x)

        x = self.mid_block1(x, t)
        x = self.mid_attn(x)
        x = self.mid_block2(x, t)

        for block1, block2, attn, upsample in self.ups:
            x = torch.cat((x, h.pop()), dim=1) #  Up은 skip connection으로 정보 보충을 받고 있음, 굳이 PixelShuffle까지 쓰면 중복
            x = block1(x, t)
            x = torch.cat((x, h.pop()), dim=1)
            x = block2(x, t)
            x = attn(x)
            x = upsample(x)

        x = torch.cat((x, r), dim=1)
        x = self.final_res_block(x, t)
        return self.final_conv(x)

class Classifier(nn.Module):
    def __init__(self, num_classes=3, img_channels=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(img_channels, 32, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(32, 32, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(32, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(256, num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        f = self.features(x)
        f = self.pool(f)
        f = torch.flatten(f, 1)
        return self.classifier(f)

    def extract_features(self, x):
        f = self.features(x)
        f = self.pool(f)
        return torch.flatten(f, 1)

## 2. 디퓨전 유틸 함수

In [2]:
timesteps = 1000
betas = torch.linspace(0.0001, 0.02, timesteps)
alphas = 1. - betas
alphas_cumprod = torch.cumprod(alphas, axis=0)
alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - alphas_cumprod)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)

def extract(a, t, x_shape):
    b = t.shape[0]
    out = a.gather(-1, t.cpu())
    return out.reshape(b, *((1,) * (len(x_shape) - 1))).to(t.device)

def q_sample(x_start, t, noise=None):
    if noise is None: noise = torch.randn_like(x_start)
    return extract(sqrt_alphas_cumprod, t, x_start.shape) * x_start + \
           extract(sqrt_one_minus_alphas_cumprod, t, x_start.shape) * noise

ddim_steps = 50
ddim_eta = 0.0  # 0 = deterministic DDIM

@torch.no_grad()
def sample_ddim(diffusion, y, device, ddim_steps, timesteps, alphas_cumprod):
    diffusion.eval()
    n = y.size(0)

    x = torch.randn((n, channels, image_size, image_size), device=device)

    ddim_ts = torch.linspace(timesteps - 1, 0, ddim_steps).long().to(device)

    for i in range(len(ddim_ts) - 1):
        t = ddim_ts[i]
        t_prev = ddim_ts[i + 1]

        t_batch = torch.full((n,), t, device=device)
        t_prev_batch = torch.full((n,), t_prev, device=device)

        eps = diffusion(x, t_batch, y)

        alpha_t = extract(alphas_cumprod, t_batch, x.shape)
        alpha_prev = extract(alphas_cumprod, t_prev_batch, x.shape)

        sqrt_alpha_t = torch.sqrt(alpha_t)
        sqrt_one_minus_alpha_t = torch.sqrt(1 - alpha_t)

        x0_pred = (x - sqrt_one_minus_alpha_t * eps) / sqrt_alpha_t
        x0_pred = x0_pred.clamp(-1.0, 1.0)

        sigma_t = torch.sqrt((1 - alpha_prev) / (1 - alpha_t)) * torch.sqrt((1 - alpha_t) / alpha_prev) # σt 계산
        sigma_t = ddim_eta
        dir_xt = torch.sqrt(1 - alpha_prev - sigma_t**2) * eps  # 수정된 방향

        random_noise = torch.randn_like(x) * sigma_t 

        x = torch.sqrt(alpha_prev) * x0_pred + dir_xt + random_noise

    return x

def denorm(x):
    return (x + 1) / 2

@torch.no_grad()
def save_ddim_samples(diffusion, device, save_dir, num_per_class, ddim_steps, timesteps, alphas_cumprod, epoch):
    diffusion.eval()
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    grids = []

    for cls in [0, 1, 2]:
        print(f"[Sampling] class {cls}")

        y = torch.full((num_per_class,), cls, device=device, dtype=torch.long)

        x = sample_ddim(diffusion, y, device, ddim_steps=ddim_steps, timesteps=timesteps, alphas_cumprod=alphas_cumprod)

        x = denorm(x).clamp(0, 1)

        grids.append(x)

    grid = make_grid(torch.cat(grids, dim=0), nrow=num_per_class)

    save_image(grid, save_dir / f"samples_epoch_{epoch:03d}.png")

    print(f"[Saved] {save_dir}")  # 마지막 그리드 이미지만 저장
    
@torch.no_grad()
def visualize_batch_real_fake(classifier, x_real, y_real, x_fake, y_fake, device, save_path=None):
    classifier.eval()

    x_real = x_real.to(device)
    feats_real = classifier.extract_features(x_real).cpu()

    if x_fake is not None:
        x_fake = x_fake.to(device)
        feats_fake = classifier.extract_features(x_fake).cpu()
    else:
        print("No fake samples in this batch.")
        return

    all_feats = torch.cat([feats_real, feats_fake]).numpy()

    tsne = TSNE(n_components=2, perplexity=20, random_state=0)
    z = tsne.fit_transform(all_feats)

    z_real = z[:len(feats_real)]
    z_fake = z[len(feats_real):]

    plt.figure(figsize=(6,5))

    colors = {0: "#4C72B0", 1: "#DD8452", 2: "#55A868"}

    for cls in [0,1,2]:
        idx = (y_real.cpu().numpy() == cls)
        plt.scatter(z_real[idx,0], z_real[idx,1], c=colors[cls], s=25, alpha=0.4, label=f"class {cls} (real)")

    for cls in [1,2]:
        idx = (y_fake.cpu().numpy() == cls)
        plt.scatter(z_fake[idx,0], z_fake[idx,1], c=colors[cls], s=60, edgecolor="black", marker="x", label=f"class {cls} (fake)")

    plt.legend()
    plt.title("Current Batch: Real vs Generated")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300)

    plt.show()

def balanced_batch(real_y, diffusion, device, verbose=False):
    n0 = (real_y == 0).sum().item()
    if n0 == 0:
        return None, None

    fake_x_list, fake_y_list = [], []

    for cls in [1, 2]:
        n_real = (real_y == cls).sum().item()
        n_missing = max(0, n0 - n_real)

        if n_missing > 0:
            if verbose:
                print(f"  -> class {cls}: generate {n_missing} fake")

            y_fake = torch.full((n_missing,), cls, device=device)
            x_fake = sample_ddim(diffusion, y_fake, device, ddim_steps=50, timesteps=timesteps, alphas_cumprod=alphas_cumprod)

            fake_x_list.append(x_fake)
            fake_y_list.append(y_fake)

    if len(fake_x_list) == 0:
        return None, None

    return torch.cat(fake_x_list, dim=0), torch.cat(fake_y_list, dim=0)

## 3. 데이터 로드

In [3]:
image_size = 64
channels = 3
batch_size = 60

data_root = r"../data/256"

transform = T.Compose([
    T.CenterCrop(64),
    T.ToTensor(),
    T.Lambda(lambda t: (t * 2) - 1)
])

dataset_full = datasets.ImageFolder(
    root=data_root,
    transform=transform
)

print("Class mapping:", dataset_full.class_to_idx)

class_indices = defaultdict(list)

for idx, label in enumerate(dataset_full.targets):
    class_indices[label].append(idx)

for cls in class_indices:
    random.shuffle(class_indices[cls])

train_counts = {
    0: 800,  # Normal (majority)
    1: 80,   # Underfill_50FR
    2: 80    # Underfill_Fan
}

test_counts = {
    0: 100,
    1: 100,
    2: 100
}

def make_split(class_indices, train_counts, test_counts):

    train_idx = []
    test_idx = []

    for cls in train_counts:

        n_train = train_counts[cls]
        n_test = test_counts.get(cls, 0)

        idxs = class_indices[cls]

        train_idx.extend(idxs[:n_train])
        test_idx.extend(idxs[n_train:n_train + n_test])

    return train_idx, test_idx

train_indices, test_indices = make_split(class_indices, train_counts, test_counts)

train_dataset = Subset(dataset_full, train_indices)
test_dataset = Subset(dataset_full, test_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

train_labels = [dataset_full.targets[i] for i in train_indices]
print("Train label distribution:", Counter(train_labels))

test_labels = [dataset_full.targets[i] for i in test_indices]
print("Test label distribution:", Counter(test_labels))

Class mapping: {'Normal': 0, 'Underfill_50FR': 1, 'Underfill_Fan': 2}
Train label distribution: Counter({0: 800, 1: 80, 2: 80})
Test label distribution: Counter({0: 100, 1: 100, 2: 100})


## 4. 평가함수

In [4]:
def evaluate_classifier(model, dataloader, device, class_names=None):
    model.eval()

    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)

            logits = model(x)
            probs = F.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_probs.append(probs.cpu())
            all_preds.append(preds.cpu())
            all_labels.append(y)

    all_probs  = torch.cat(all_probs).numpy()
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_preds, average=None, zero_division=0)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)

    cm = confusion_matrix(all_labels, all_preds)

    print("===== Evaluation =====")
    print(f"Accuracy        : {acc:.4f}")
    print(f"Precision(macro): {precision_macro:.4f}")
    print(f"Recall(macro)   : {recall_macro:.4f}")
    print(f"F1(macro)       : {f1_macro:.4f}")
    print("\nPer-class:")
    if class_names is None:
        class_names = [str(i) for i in range(len(precision))]

    for i, cls in enumerate(class_names):
        print(f"Class {cls} | " f"P:{precision[i]:.3f} " f"R:{recall[i]:.3f} " f"F1:{f1[i]:.3f} " f"N:{support[i]}")

    print("\nConfusion Matrix:\n", cm)

    return { "accuracy": acc, "precision": precision, "recall": recall, "f1": f1, "support": support, "confusion_matrix": cm, "probs": all_probs, "preds": all_preds, "labels": all_labels}

def metrics_to_rows(metrics, seed):
    rows = []

    num_classes = len(metrics["f1"])

    for cls in range(num_classes):
        rows.append({"seed": seed, "class": cls, "precision": metrics["precision"][cls], "recall": metrics["recall"][cls], "f1": metrics["f1"][cls], "support": metrics["support"][cls], "accuracy": metrics["accuracy"]})

    rows.append({"seed": seed, "class": "macro", "precision": metrics["precision"].mean(), "recall": metrics["recall"].mean(), "f1": metrics["f1"].mean(), "support": metrics["support"].sum(), "accuracy": metrics["accuracy"]})

    return rows

## 5. baseline

In [5]:
def run_experiment(seed):
    set_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    classifier = Classifier(num_classes=3).to(device)

    opt_c = torch.optim.Adam(classifier.parameters(), lr=1e-3)
    cls_loss_fn = nn.CrossEntropyLoss()

    epochs = 201
    cls_loss_history = []

    for epoch in range(0, epochs):

        classifier.train()
        epoch_cls_losses = []

        pbar = tqdm(train_loader, desc=f"[Seed {seed}] Epoch {epoch:02d}/{epochs-1}",leave=False)

        for x, y in pbar:
            x = x.to(device)
            y = y.to(device)

            logits_real = classifier(x)
            loss_real = cls_loss_fn(logits_real, y)

            loss_cls = loss_real

            opt_c.zero_grad()
            loss_cls.backward()
            opt_c.step()

            epoch_cls_losses.append(loss_cls.item())

            pbar.set_postfix(cls=f"{loss_cls.item():.4f}")

        cls_loss_history.append(np.mean(epoch_cls_losses))

        print(f"[Seed {seed} | Epoch {epoch:02d}] " f"Cls={cls_loss_history[-1]:.4f} | ")

    metrics = evaluate_classifier(classifier, test_loader, device, ["0", "1", "2"])

    return metrics

all_rows = []

for seed in range(10):
    print(f"\n===== Seed {seed} =====")
    metrics = run_experiment(seed)

    rows = metrics_to_rows(metrics, seed)
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)
save_path = "results/baseline_diffusion.xlsx"
os.makedirs("results", exist_ok=True)
df.to_excel(save_path, index=False)


===== Seed 0 =====


## 6. 디퓨전 생성 모델

In [6]:
import os

def run_experiment(seed):
    set_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    diffusion = Unet(dim=image_size, channels=channels, num_classes=3).to(device)

    opt_g = torch.optim.Adam(diffusion.parameters(), lr=1e-3)

    diffusion_loss_fn = nn.MSELoss()

    epochs = 501

    diff_loss_history = []

    for epoch in range(0, epochs):
        diffusion.train()

        epoch_diff_losses = []

        pbar = tqdm(train_loader, desc=f"[Seed {seed}] Epoch {epoch:02d}/{epochs-1}", leave=False)

        for x, y in pbar:
            x = x.to(device)
            y = y.to(device)

            t = torch.randint(0, timesteps, (x.size(0),), device=device)
            noise = torch.randn_like(x)
            x_noisy = q_sample(x, t, noise)
            pred_noise = diffusion(x_noisy, t, y)

            loss_diffusion = diffusion_loss_fn(pred_noise, noise)

            opt_g.zero_grad()
            loss_diffusion.backward()
            opt_g.step()

            epoch_diff_losses.append(loss_diffusion.item())

            pbar.set_postfix(diff=f"{loss_diffusion.item():.4f}")

        diff_loss_history.append(np.mean(epoch_diff_losses))

        print(f"[Seed {seed} | Epoch {epoch:02d}] "f"Diff={diff_loss_history[-1]:.4f} | ")

        if epoch % 10 == 0:
            save_ddim_samples(diffusion=diffusion, device=device, save_dir="samples/diffusion_only", num_per_class=10, ddim_steps=20, timesteps=timesteps, alphas_cumprod=alphas_cumprod, epoch=epoch)

    os.makedirs("saved_models", exist_ok=True)
    save_path = os.path.join("saved_models", f"diffusion_model.pth")
    torch.save(diffusion.state_dict(), save_path)

for seed in range(1):
    print(f"\n===== Seed {seed} =====")
    run_experiment(seed)


===== Seed 0 =====


## 7. conditional diffusion

In [ ]:
def run_experiment(seed):
    set_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    diffusion = Unet(dim=image_size, channels=channels, num_classes=3).to(device)
    classifier = Classifier(num_classes=3).to(device)

    opt_g = torch.optim.Adam(diffusion.parameters(), lr=1e-3)
    opt_c = torch.optim.Adam(classifier.parameters(), lr=1e-3)

    diffusion_loss_fn = nn.MSELoss()
    cls_loss_fn = nn.CrossEntropyLoss()

    epochs = 201

    diff_loss_history = []
    cls_loss_history = []

    for epoch in range(1, epochs):
        diffusion.train()

        epoch_diff_losses = []

        pbar = tqdm(train_loader, desc=f"[Seed {seed}] Diffusion Epoch {epoch:02d}/{epochs-1}", leave=False)

        for x, y in pbar:
            x = x.to(device)
            y = y.to(device)

            t = torch.randint(0, timesteps, (x.size(0),), device=device)
            noise = torch.randn_like(x)
            x_noisy = q_sample(x, t, noise)

            pred_noise = diffusion(x_noisy, t, y)

            loss_diffusion = diffusion_loss_fn(pred_noise, noise)

            opt_g.zero_grad()
            loss_diffusion.backward()
            opt_g.step()

            epoch_diff_losses.append(loss_diffusion.item())

            pbar.set_postfix(diff=f"{loss_diffusion.item():.4f}")

        diff_loss_history.append(np.mean(epoch_diff_losses))

        print(f"[Seed {seed} | Diffusion Epoch {epoch:02d}] Diff={diff_loss_history[-1]:.4f}")

    print("\n===== Generating Fake Data for class 1 and 2 =====")
    fake_data = {1: [], 2: []}
    
    for cls in [1, 2]:
        y_fake = torch.full((720,), cls, dtype=torch.long, device=device)
        fake_samples = sample_ddim(
            diffusion=diffusion,
            y=y_fake,
            device=device,
            ddim_steps=50,
            timesteps=timesteps,  # timesteps 추가
            alphas_cumprod=alphas_cumprod 
        )
        fake_data[cls] = fake_samples.cpu()

    print("\n===== Saving Fake Data for class 1 and 2 =====")
    for cls in [1, 2]:
        class_dir = os.path.join("samples/conditional", f'class_{cls}')
        os.makedirs(class_dir, exist_ok=True)  # 폴더가 없으면 생성

        for i, img in enumerate(fake_data[cls]):
            save_path = os.path.join(class_dir, f"{i:03d}.png")
            save_image(img, save_path)
        print(f"Saved {len(fake_data[cls])} fake images to {class_dir}")

    print("\n===== Training Classifier =====")
    for epoch in range(1, epochs):
        classifier.train()
        epoch_cls_losses = []

        pbar = tqdm(train_loader, desc=f"[Seed {seed}] Classifier Epoch {epoch:02d}/{epochs-1}", leave=False)

        for x, y in pbar:
            x = x.to(device)
            y = y.to(device)

            num_class0 = (y == 0).sum().item()
            num_class1 = (y == 1).sum().item()
            num_class2 = (y == 2).sum().item()

            total_class0 = num_class0  # class 0에 맞춰서 class 1, class 2의 수를 맞출 예정

            fake_class1_needed = total_class0 - num_class1
            fake_class2_needed = total_class0 - num_class2

            if fake_class1_needed > 0:
                idx_class1 = torch.randint(0, len(fake_data[1]), (fake_class1_needed,))
                fake_class1_batch = fake_data[1][idx_class1].to(device)

            if fake_class2_needed > 0:
                idx_class2 = torch.randint(0, len(fake_data[2]), (fake_class2_needed,))
                fake_class2_batch = fake_data[2][idx_class2].to(device)

            real_data_class0 = x[y == 0]
            real_labels_class0 = y[y == 0]

            real_data_class1 = x[y == 1]
            real_labels_class1 = y[y == 1]

            real_data_class2 = x[y == 2]
            real_labels_class2 = y[y == 2]

            x_aug = torch.cat([real_data_class0, real_data_class1, real_data_class2, fake_class1_batch, fake_class2_batch], dim=0)

            y_aug = torch.cat([real_labels_class0, real_labels_class1, real_labels_class2, torch.full((fake_class1_batch.size(0),), 1, dtype=torch.long, device=device),torch.full((fake_class2_batch.size(0),), 2, dtype=torch.long, device=device)], dim=0)

            logits_real = classifier(x_aug)
            loss_cls = cls_loss_fn(logits_real, y_aug)

            opt_c.zero_grad()
            loss_cls.backward()
            opt_c.step()

            epoch_cls_losses.append(loss_cls.item())

            pbar.set_postfix(cls=f"{loss_cls.item():.4f}")

        cls_loss_history.append(np.mean(epoch_cls_losses))

        print(f"[Seed {seed} | Classifier Epoch {epoch:02d}] Cls={cls_loss_history[-1]:.4f}")

    metrics = evaluate_classifier(classifier, test_loader, device, ["0", "1", "2"])

    return metrics

all_rows = []

for seed in range(10):
    print(f"\n===== Seed {seed} =====")
    metrics = run_experiment(seed)

    rows = metrics_to_rows(metrics, seed)
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)

save_path = "results/conditional_diffusion.xlsx"
os.makedirs("results", exist_ok=True)
df.to_excel(save_path, index=False)

print(f"Saved metrics to: {save_path}")

## 8. joint diffusion

In [6]:
def run_experiment(seed):
    set_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    diffusion = Unet(dim=image_size, channels=channels, num_classes=3).to(device)
    classifier = Classifier(num_classes=3).to(device)

    opt_g = torch.optim.Adam(diffusion.parameters(), lr=1e-3)
    opt_c = torch.optim.Adam(classifier.parameters(), lr=1e-3)

    diffusion_loss_fn = nn.MSELoss()
    cls_loss_fn = nn.CrossEntropyLoss()

    epochs = 201
    lambda_start = 1.0

    diff_loss_history = []
    cls_loss_history = []

    for epoch in range(epochs):
        lambda_t = lambda_start * (1 - epoch / (epochs - 1))

        diffusion.train()
        classifier.train()

        epoch_diff_losses = []
        epoch_cls_losses = []

        pbar = tqdm(train_loader, desc=f"[Seed {seed}] Epoch {epoch:02d}/{epochs-1}", leave=False)

        for x, y in pbar:
            x = x.to(device)
            y = y.to(device)

            t = torch.randint(0, timesteps, (x.size(0),), device=device)
            noise = torch.randn_like(x)
            x_noisy = q_sample(x, t, noise)
            pred_noise = diffusion(x_noisy, t, y)

            loss_diff = diffusion_loss_fn(pred_noise, noise)

            x_fake, y_fake = balanced_batch(y, diffusion, device)

            if x_fake is not None:
                logits_fake_c = classifier(x_fake.detach())
                loss_fake_c = cls_loss_fn(logits_fake_c, y_fake)

                logits_fake_g = classifier(x_fake)
                loss_fake_g = cls_loss_fn(logits_fake_g, y_fake)
            else:
                loss_fake_c = torch.tensor(0.0, device=device)
                loss_fake_g = torch.tensor(0.0, device=device)

            loss_diffusion = (loss_diff + (1 - lambda_t) * loss_fake_g)
            opt_g.zero_grad()
            loss_diffusion.backward()
            opt_g.step()

            logits_real = classifier(x)
            loss_real = cls_loss_fn(logits_real, y)

            loss_cls = loss_real + (1 - lambda_t) * loss_fake_c

            opt_c.zero_grad()
            loss_cls.backward()
            opt_c.step()

            epoch_diff_losses.append(loss_diffusion.item())
            epoch_cls_losses.append(loss_cls.item())

            pbar.set_postfix(diff=f"{loss_diffusion.item():.4f}", cls=f"{loss_cls.item():.4f}", lam=f"{lambda_t:.3f}")

        if epoch % 50 == 0:
            save_ddim_samples(diffusion=diffusion, device=device, save_dir="samples/joint", num_per_class=10, ddim_steps=50, timesteps=timesteps, alphas_cumprod=alphas_cumprod, epoch=epoch)

        diff_loss_history.append(np.mean(epoch_diff_losses))
        cls_loss_history.append(np.mean(epoch_cls_losses))

        print(f"[Seed {seed} | Epoch {epoch:02d}] " f"Diff={diff_loss_history[-1]:.4f} | " f"Cls={cls_loss_history[-1]:.4f} | " f"λ={lambda_t:.3f}")

    metrics = evaluate_classifier(classifier, test_loader, device, ["0", "1", "2"])
    return metrics

all_rows = []

for seed in range(10):
    print(f"\n===== Seed {seed} =====")
    metrics = run_experiment(seed)
    rows = metrics_to_rows(metrics, seed)
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)
save_path = "results/joint_diffusion.xlsx"
os.makedirs("results", exist_ok=True)
df.to_excel(save_path, index=False)


===== Seed 0 =====


[Sampling] class 0
[Sampling] class 1
[Sampling] class 2
[Saved] samples/joint
[Seed 0 | Epoch 00] Diff=1.0580 | Cls=0.7269 | λ=1.000


[Seed 0 | Epoch 01] Diff=0.6648 | Cls=0.6035 | λ=0.995


[Seed 0 | Epoch 02] Diff=0.2086 | Cls=0.5812 | λ=0.990


[Seed 0 | Epoch 03] Diff=0.1043 | Cls=0.5859 | λ=0.985


...

[Seed 0 | Epoch 196] Diff=0.3047 | Cls=0.4134 | λ=0.020


[Seed 0 | Epoch 197] Diff=0.2658 | Cls=0.3662 | λ=0.015


[Seed 0 | Epoch 198] Diff=0.3724 | Cls=0.4746 | λ=0.010


[Seed 0 | Epoch 199] Diff=0.5372 | Cls=0.6799 | λ=0.005


[Sampling] class 0
[Sampling] class 1
[Sampling] class 2
[Saved] samples/joint
[Seed 0 | Epoch 200] Diff=0.3845 | Cls=0.4849 | λ=0.000
===== Evaluation =====
Accuracy        : 0.6533
Precision(macro): 0.6605
Recall(macro)   : 0.6533
F1(macro)       : 0.6319

Per-class:
Class 0 | P:0.641 R:1.000 F1:0.781 N:100
Class 1 | P:0.642 R:0.520 F1:0.575 N:100
Class 2 | P:0.698 R:0.440 F1:0.540 N:100

Confusion Matrix:
 [[100   0   0]
 [ 29  52  19]
 [ 27  29  44]]


## 9. Decision Boundary

In [7]:
def run_experiment(seed):
    set_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    diffusion = Unet(dim=image_size, channels=channels, num_classes=3).to(device)
    classifier = Classifier(num_classes=3).to(device)

    opt_g = torch.optim.Adam(diffusion.parameters(), lr=1e-3)
    opt_c = torch.optim.Adam(classifier.parameters(), lr=1e-3)

    diffusion_loss_fn = nn.MSELoss()
    cls_loss_fn = nn.CrossEntropyLoss()

    epochs = 201
    lambda_start = 1.0

    diff_loss_history = []
    cls_loss_history = []

    for epoch in range(1, epochs):
        lambda_t = lambda_start * (1 - epoch / (epochs - 1))

        diffusion.train()
        classifier.train()

        epoch_diff_losses = []
        epoch_cls_losses = []

        pbar = tqdm(train_loader, desc=f"[Seed {seed}] Epoch {epoch:02d}/{epochs-1}", leave=False)

        for x, y in pbar:
            x = x.to(device)
            y = y.to(device)

            t = torch.randint(0, timesteps, (x.size(0),), device=device)
            noise = torch.randn_like(x)
            x_noisy = q_sample(x, t, noise)
            pred_noise = diffusion(x_noisy, t, y)

            loss_diff = diffusion_loss_fn(pred_noise, noise)

            x0_pred = (x_noisy- extract(sqrt_one_minus_alphas_cumprod, t, x.shape) * pred_noise) / extract(sqrt_alphas_cumprod, t, x.shape)
            
            logits_bd = classifier(x0_pred)   # detach 제거
            probs = F.softmax(logits_bd, dim=1)

            minority_mask = (y != 0)
            p_mi = probs[torch.arange(len(y), device=y.device), y]
            p_ma = probs[:, 0]

            boundary_mask = minority_mask & (p_mi > p_ma)
            loss_boundary = ((p_mi[boundary_mask] - p_ma[boundary_mask]) ** 2).mean() \
                if boundary_mask.any() else torch.zeros((), device=x0_pred.device, dtype=x0_pred.dtype)

            x_fake, y_fake = balanced_batch(y, diffusion, device)

            if x_fake is not None:
                logits_fake_c = classifier(x_fake.detach())
                loss_fake_c = cls_loss_fn(logits_fake_c, y_fake)

                logits_fake_g = classifier(x_fake)
                loss_fake_g = cls_loss_fn(logits_fake_g, y_fake)
            else:
                loss_fake_c = torch.tensor(0.0, device=device)
                loss_fake_g = torch.tensor(0.0, device=device)

            loss_diffusion = (loss_diff + lambda_t * loss_boundary + (1 - lambda_t) * loss_fake_g)
            opt_g.zero_grad()
            loss_diffusion.backward()
            opt_g.step()

            logits_real = classifier(x)
            loss_real = cls_loss_fn(logits_real, y)

            loss_cls = loss_real + (1 - lambda_t) * loss_fake_c

            opt_c.zero_grad()
            loss_cls.backward()
            opt_c.step()

            epoch_diff_losses.append(loss_diffusion.item())
            epoch_cls_losses.append(loss_cls.item())

            pbar.set_postfix(diff=f"{loss_diffusion.item():.4f}", cls=f"{loss_cls.item():.4f}", bd=f"{loss_boundary.item():.4f}", lam=f"{lambda_t:.3f}")

        diff_loss_history.append(np.mean(epoch_diff_losses))
        cls_loss_history.append(np.mean(epoch_cls_losses))

        print(f"[Seed {seed} | Epoch {epoch:02d}] " f"Diff={diff_loss_history[-1]:.4f} | " f"Cls={cls_loss_history[-1]:.4f} | " f"λ={lambda_t:.3f}")

        if epoch % 50 == 0:
            save_ddim_samples(diffusion=diffusion, device=device, save_dir="samples/decision_boundary", num_per_class=10, ddim_steps=50, timesteps=timesteps, alphas_cumprod=alphas_cumprod, epoch=epoch)

    metrics = evaluate_classifier(classifier, test_loader, device, ["0", "1", "2"])
    return metrics

all_rows = []

for seed in range(10):
    print(f"\n===== Seed {seed} =====")
    metrics = run_experiment(seed)
    rows = metrics_to_rows(metrics, seed)
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)
save_path = "results/decisionboundary_diffusion.xlsx"
os.makedirs("results", exist_ok=True)
df.to_excel(save_path, index=False)


===== Seed 0 =====


[Seed 0 | Epoch 01] Diff=1.1765 | Cls=0.7332 | λ=0.995


[Seed 0 | Epoch 02] Diff=0.7776 | Cls=0.6087 | λ=0.990


[Seed 0 | Epoch 03] Diff=0.4563 | Cls=0.5872 | λ=0.985


[Seed 0 | Epoch 04] Diff=0.2605 | Cls=0.5855 | λ=0.980


...

[Seed 0 | Epoch 196] Diff=nan | Cls=nan | λ=0.020


[Seed 0 | Epoch 197] Diff=nan | Cls=nan | λ=0.015


[Seed 0 | Epoch 198] Diff=nan | Cls=nan | λ=0.010


[Seed 0 | Epoch 199] Diff=nan | Cls=nan | λ=0.005


[Seed 0 | Epoch 200] Diff=nan | Cls=nan | λ=0.000
[Sampling] class 0
[Sampling] class 1
[Sampling] class 2
[Saved] samples/decision_boundary
===== Evaluation =====
Accuracy        : 0.3333
Precision(macro): 0.1111
Recall(macro)   : 0.3333
F1(macro)       : 0.1667

Per-class:
Class 0 | P:0.333 R:1.000 F1:0.500 N:100
Class 1 | P:0.000 R:0.000 F1:0.000 N:100
Class 2 | P:0.000 R:0.000 F1:0.000 N:100

Confusion Matrix:
 [[100   0   0]
 [100   0   0]
 [100   0   0]]


## 10. 통계

In [9]:
import itertools
import pandas as pd
from scipy.stats import ttest_rel

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

paths = {
    "baseline": "results/baseline_diffusion.xlsx",
    "conditional_diffusion": "results/conditional_diffusion.xlsx",
    "joint_diffusion": "results/joint_diffusion.xlsx",
    "decisionboundary_diffusion": "results/decisionboundary_diffusion.xlsx",
}

dfs = {
    name: pd.read_excel(path)
    for name, path in paths.items()
}

def get_metric(df, metric="f1", class_label="macro"):
    return (
        df[df["class"] == class_label]
        .sort_values("seed")[metric]
        .values
    )

metrics = ["precision", "recall", "f1"]

results = []

for (name_a, df_a), (name_b, df_b) in itertools.combinations(dfs.items(), 2):
    for metric in metrics:
        scores_a = get_metric(df_a, metric=metric, class_label="macro")
        scores_b = get_metric(df_b, metric=metric, class_label="macro")

        assert len(scores_a) == len(scores_b), "Seed 수 다름"

        t_stat, p_value = ttest_rel(scores_a, scores_b)

        results.append({
            "method_A": name_a,
            "method_B": name_b,
            "metric": metric,
            "mean_A": scores_a.mean(),
            "mean_B": scores_b.mean(),
            "mean_diff(B-A)": scores_b.mean() - scores_a.mean(),
            "t_stat": t_stat,
            "p_value": p_value,
            "significant(p<0.05)": p_value < 0.05
        })

ttest_df = pd.DataFrame(results)
ttest_df

## 11. 평균

In [ ]:
metrics = ["precision", "recall", "f1"]

results = []

for name, df in dfs.items():
    row = {"method": name}

    for metric in metrics:
        scores = (
            df[df["class"] == "macro"]
            .sort_values("seed")[metric]
            .values
        )
        row[f"{metric}_mean"] = scores.mean()

    results.append(row)

mean_df = pd.DataFrame(results)
mean_df